In [3]:
import pandas as pd

xl = pd.ExcelFile('/Users/av/Desktop/255/255-AI/data/raw/felten/AIOE_DataAppendix.xlsx')
print("Sheets:", xl.sheet_names)

# Preview every sheet
for sheet in xl.sheet_names:
    df = xl.parse(sheet, nrows=3, dtype=str)
    print(f"\n── Sheet: {sheet} ──────────────")
    print("Columns:", df.columns.tolist())
    print(df.head(3).to_string())

Sheets: ['Index', 'Appendix A', 'Appendix B', 'Appendix C', 'Appendix D', 'Appendix E']

── Sheet: Index ──────────────
Columns: ['Data Appendix A: AIOE Scores by Occupation']
                    Data Appendix A: AIOE Scores by Occupation
0     Data Appendix B: AIIE Scores by Industry (4-Digit NAICS)
1                    Data Appendix C: AIGE Scores by FIPS Code
2  Data Appendix D: AI Application-Occupational Ability Matrix

── Sheet: Appendix A ──────────────
Columns: ['SOC Code', 'Occupation Title', 'AIOE']
  SOC Code                     Occupation Title       AIOE
0  11-1011                     Chief Executives   1.334246
1  11-1021      General and Operations Managers  0.5748773
2  11-2011  Advertising and Promotions Managers   1.294387

── Sheet: Appendix B ──────────────
Columns: ['NAICS', 'Industry Title', 'AIIE']
  NAICS                            Industry Title       AIIE
0  1133                                   Logging  -1.360161
1  1151    Support Activities for Crop Produc

In [8]:
import pandas as pd
from scipy import stats
import os

# ── 1. Load Appendix A only ─────────────────────────────────────
felten = pd.read_excel(
    '/Users/av/Desktop/255/255-AI/data/raw/felten/AIOE_DataAppendix.xlsx',
    sheet_name='Appendix A',
    dtype={'SOC Code': str}
)

print("Raw shape:", felten.shape)
print("Columns:", felten.columns.tolist())
print(felten.head(3).to_string())

# ── 2. Rename columns to standard names ────────────────────────
felten = felten.rename(columns={
    'SOC Code':         'OCC_CODE',
    'Occupation Title': 'OCC_TITLE_FELTEN',
    'AIOE':             'AIOE'
})

# ── 3. Standardize SOC code format ─────────────────────────────
felten['OCC_CODE'] = felten['OCC_CODE'].str.strip()

# Confirm format
format_ok = felten['OCC_CODE'].str.match(r'^\d{2}-\d{4}$').all()
print(f"\nSOC format check: {format_ok}")
if not format_ok:
    bad = felten[~felten['OCC_CODE'].str.match(r'^\d{2}-\d{4}$')]
    print("Bad format rows:")
    print(bad.head())

# ── 4. Check for missing values ─────────────────────────────────
print(f"\nMissing values:\n{felten.isnull().sum()}")

# ── 5. Check for duplicates ─────────────────────────────────────
dups = felten.duplicated(subset='OCC_CODE').sum()
print(f"\nDuplicate SOC codes: {dups}")

# ── 6. Add z-score of AIOE ──────────────────────────────────────
felten['AIOE_z'] = stats.zscore(felten['AIOE'])

# ── 7. Check score distribution ─────────────────────────────────
print(f"\nAIOE score distribution:")
print(felten['AIOE'].describe().round(3))

# ── 8. Keep only needed columns ─────────────────────────────────
felten_clean = felten[['OCC_CODE', 'OCC_TITLE_FELTEN', 'AIOE', 'AIOE_z']].copy()

# ── 9. Spot checks ───────────────────────────────────────────────
print("\nSpot check — Software Developers (15-1252):")
print(felten_clean[felten_clean['OCC_CODE'] == '15-1252'].to_string())

print("\nTop 5 highest AIOE occupations:")
print(felten_clean.nlargest(5, 'AIOE')[['OCC_CODE', 'OCC_TITLE_FELTEN', 'AIOE']].to_string())

print("\nBottom 5 lowest AIOE occupations:")
print(felten_clean.nsmallest(5, 'AIOE')[['OCC_CODE', 'OCC_TITLE_FELTEN', 'AIOE']].to_string())

# ── 10. Save ─────────────────────────────────────────────────────
os.makedirs('data/processed/felten', exist_ok=True)
felten_clean.to_csv('data/processed/felten/felten_cleaned.csv', index=False)
print(f"\nSaved: data/processed/felten/felten_cleaned.csv")
print(f"Final shape: {felten_clean.shape}")

Raw shape: (774, 3)
Columns: ['SOC Code', 'Occupation Title', 'AIOE']
  SOC Code                     Occupation Title      AIOE
0  11-1011                     Chief Executives  1.334246
1  11-1021      General and Operations Managers  0.574877
2  11-2011  Advertising and Promotions Managers  1.294387

SOC format check: True

Missing values:
OCC_CODE            0
OCC_TITLE_FELTEN    0
AIOE                0
dtype: int64

Duplicate SOC codes: 0

AIOE score distribution:
count    774.000
mean      -0.000
std        1.000
min       -2.670
25%       -0.865
50%       -0.051
75%        1.014
max        1.528
Name: AIOE, dtype: float64

Spot check — Software Developers (15-1252):
Empty DataFrame
Columns: [OCC_CODE, OCC_TITLE_FELTEN, AIOE, AIOE_z]
Index: []

Top 5 highest AIOE occupations:
    OCC_CODE                                                OCC_TITLE_FELTEN      AIOE
335  29-9092                                              Genetic Counselors  1.527667
59   13-2061                       